# Exploratory Data Analysis
### Stock Market Index Movement Forecasting Using Multiple Indicators and Deep Learning

This notebook demonstrates the core data pipeline on NIFTY 50 data: loading, cleaning, indicator calculation, feature selection, scaling, and sequence generation. It mirrors the logic in `src/` so you can inspect intermediate results interactively.

> Run this notebook from the project root (or adjust `sys.path` below) so the `src` and `config` packages can be imported.

In [ ]:
import sys
from pathlib import Path

# Ensure the project root is on sys.path when running from notebooks/
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import matplotlib.pyplot as plt

from config import settings
from src.preprocessing import clean_ohlcv
from src.feature_engineering import build_feature_dataframe, fit_scaler_on_training_data
from src.sequences import create_sequences

pd.set_option("display.max_columns", None)

## 1. Load NIFTY 50 Data
Uses the bundled sample dataset directly for a fast, offline-friendly demo. Replace with `src.data_loader.load_market_data` for a live download.

In [ ]:
sample_path = settings.SAMPLE_DATA_DIR / settings.INDEX_OPTIONS["NIFTY50"]["sample_csv"]
raw_df = pd.read_csv(sample_path, parse_dates=["Date"])
print(raw_df.shape)
raw_df.head()

## 2. Dataframe Preview & Missing Values

In [ ]:
print(raw_df.info())
print("\nMissing values per column:")
print(raw_df.isna().sum())

## 3. Clean the Data
Sorting, de-duplication, and missing-value handling (see `src/preprocessing.py`).

In [ ]:
clean_df = clean_ohlcv(raw_df)
print("Rows after cleaning:", len(clean_df))
clean_df.head()

## 4. Closing Price Plot

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(clean_df["Date"], clean_df["Close"])
plt.title("NIFTY 50 - Closing Price (Sample Data)")
plt.xlabel("Date")
plt.ylabel("Close (index points)")
plt.tight_layout()
plt.show()

## 5. Indicator Calculation
SMA(20/50), EMA(12/26), MACD(12,26,9), RSI(14), plus optional extras (see `src/indicators.py`).

In [ ]:
feature_df = build_feature_dataframe(clean_df)
print("Rows after indicator warm-up removal:", len(feature_df))
feature_df[["Date", "Close", "SMA_20", "SMA_50", "EMA_12", "EMA_26", "MACD", "MACD_SIGNAL", "RSI_14"]].tail()

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)

axes[0].plot(feature_df["Date"], feature_df["Close"], label="Close")
axes[0].plot(feature_df["Date"], feature_df["SMA_20"], label="SMA 20")
axes[0].plot(feature_df["Date"], feature_df["SMA_50"], label="SMA 50")
axes[0].set_title("Price with Moving Averages")
axes[0].legend()

axes[1].plot(feature_df["Date"], feature_df["MACD"], label="MACD")
axes[1].plot(feature_df["Date"], feature_df["MACD_SIGNAL"], label="Signal")
axes[1].bar(feature_df["Date"], feature_df["MACD_HIST"], label="Histogram", alpha=0.3)
axes[1].set_title("MACD")
axes[1].legend()

axes[2].plot(feature_df["Date"], feature_df["RSI_14"], label="RSI 14", color="purple")
axes[2].axhline(70, color="red", linestyle="--", linewidth=1)
axes[2].axhline(30, color="green", linestyle="--", linewidth=1)
axes[2].set_title("RSI")
axes[2].legend()

plt.tight_layout()
plt.show()

## 6. Feature Selection
The final multivariate feature set used to train the LSTM model (see `config/settings.py: FEATURE_COLUMNS`).

In [ ]:
print(settings.FEATURE_COLUMNS)
feature_df[settings.FEATURE_COLUMNS].describe()

## 7. Chronological Split & Scaling
The scaler is fit ONLY on the training split to avoid data leakage.

In [ ]:
scaler, split = fit_scaler_on_training_data(feature_df)
print("Train / Val / Test sizes:", len(split.train), len(split.val), len(split.test))

train_scaled = scaler.transform(split.train[settings.FEATURE_COLUMNS].values)
val_scaled = scaler.transform(split.val[settings.FEATURE_COLUMNS].values)
test_scaled = scaler.transform(split.test[settings.FEATURE_COLUMNS].values)

print("Scaled train range:", train_scaled.min(), "to", train_scaled.max())

## 8. Sequence Generation
Sliding windows of `lookback` trading days become the LSTM's input tensors.

In [ ]:
target_idx = settings.FEATURE_COLUMNS.index(settings.TARGET_COLUMN)
X_train, y_train = create_sequences(train_scaled, target_idx, lookback=settings.DEFAULT_LOOKBACK)
X_val, y_val = create_sequences(val_scaled, target_idx, lookback=settings.DEFAULT_LOOKBACK)
X_test, y_test = create_sequences(test_scaled, target_idx, lookback=settings.DEFAULT_LOOKBACK)

print("X_train shape:", X_train.shape)
print("X_val shape:", X_val.shape)
print("X_test shape:", X_test.shape)

## Next Steps

Model building, training, and evaluation are implemented in `src/model.py`, `src/training.py`, and `src/evaluation.py`, and are exposed interactively through the **Model Training** and **Model Performance** pages of the Streamlit app (`streamlit run app.py` from the project root).